Import libraries:

In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

Pull the data from Postgres database:

In [3]:
load_dotenv()

user = os.environ.get('AACT_DB_USER')
password = os.environ.get('AACT_DB_PASSWORD')
host = os.environ.get('AACT_DB_HOST')
port = os.environ.get('AACT_DB_PORT')
dbname = os.environ.get('AACT_DB_NAME')

engine = create_engine(f'postgresql://{user}:{password}@{host}:{port}/{dbname}')

df = pd.read_sql("SELECT COUNT(*) FROM ctgov.studies;", engine)
print(df)

    count
0  598314


Based off of the provided data dictionary (https://aact.ctti-clinicaltrials.org/data_dictionary), select these attributes:
- nct_id: Unique trial identifier
- phase: Trial phase (1, 2, 3, 4)
- enrollment: Target number of participants a trial aims to recruit
- overall_status: Trial completion status
- start_date: Start of trial
- completion_date: End of trial
- agency_class: Trial lead sponsor type
- num_sites: How many locations the trial ran across
- num_conditions: Number of distinct conditions/diseases the trial targets
- num_interventions: Number of distinct treatment arms
- num_countries: Number of countries the trial ran across
- allocation: Randomized vs. non-randomized design
- intervention_model: Type of intervention model
- masking: Number of parties blinded
- primary_purpose: Purpose of trial
- gender, minimum_age, maximum_age, healthy_volunteers: Eligibility criteria

In [20]:
query = """
SELECT 
    s.nct_id,
    s.phase,
    s.enrollment,
    s.overall_status,
    s.start_date,
    s.completion_date,
    sp.agency_class AS sponsor_type,
    COUNT(DISTINCT f.id) AS num_sites,
    COUNT(DISTINCT c.id) AS num_conditions,
    COUNT(DISTINCT i.id) AS num_interventions,
    COUNT(DISTINCT co.name) AS num_countries,
    d.allocation,
    d.intervention_model,
    d.masking,
    d.primary_purpose,
    e.gender,
    e.minimum_age,
    e.maximum_age,
    e.healthy_volunteers
FROM ctgov.studies s
LEFT JOIN ctgov.sponsors sp 
    ON s.nct_id = sp.nct_id AND sp.lead_or_collaborator = 'lead'
LEFT JOIN ctgov.facilities f 
    ON s.nct_id = f.nct_id
LEFT JOIN ctgov.conditions c 
    ON s.nct_id = c.nct_id
LEFT JOIN ctgov.interventions i 
    ON s.nct_id = i.nct_id
LEFT JOIN ctgov.countries co 
    ON s.nct_id = co.nct_id
LEFT JOIN ctgov.designs d 
    ON s.nct_id = d.nct_id
LEFT JOIN ctgov.eligibilities e 
    ON s.nct_id = e.nct_id
WHERE s.overall_status = 'COMPLETED'
  AND s.study_type = 'INTERVENTIONAL'
  AND s.start_date IS NOT NULL
  AND s.completion_date IS NOT NULL
GROUP BY s.nct_id, s.phase, s.enrollment, s.overall_status, 
         s.start_date, s.completion_date, sp.agency_class,
         d.allocation, d.intervention_model, d.masking, d.primary_purpose,
         e.gender, e.minimum_age, e.maximum_age, e.healthy_volunteers
"""

df = pd.read_sql(query, engine)

In [21]:
df.head()

,nct_id,phase,enrollment,overall_status,start_date,completion_date,sponsor_type,num_sites,num_conditions,num_interventions,num_countries,allocation,intervention_model,masking,primary_purpose,gender,minimum_age,maximum_age,healthy_volunteers
0,NCT00000113,PHASE3,469.0,COMPLETED,1997-09-30,2013-09-30,OTHER,4,1,2,1,RANDOMIZED,PARALLEL,TRIPLE,TREATMENT,ALL,6 Years,12 Years,False
1,NCT00000114,PHASE3,NaN,COMPLETED,1984-05-31,1987-06-30,NIH,0,1,2,0,RANDOMIZED,FACTORIAL,DOUBLE,TREATMENT,ALL,18 Years,49 Years,None
2,NCT00000115,PHASE2,NaN,COMPLETED,1990-12-31,1994-06-30,NIH,0,1,1,0,RANDOMIZED,CROSSOVER,DOUBLE,TREATMENT,ALL,8 Years,NaN,None
3,NCT00000116,PHASE3,221.0,COMPLETED,1996-05-31,2002-09-30,NIH,1,1,3,1,RANDOMIZED,PARALLEL,QUADRUPLE,TREATMENT,ALL,18 Years,55 Years,True
4,NCT00000117,PHASE3,NaN,COMPLETED,1995-08-31,1997-12-31,NIH,2,1,1,1,RANDOMIZED,NaN,DOUBLE,TREATMENT,ALL,NaN,50 Years,None


Construct target variable:

In [22]:
df['start_date'] = pd.to_datetime(df['start_date'])
df['completion_date'] = pd.to_datetime(df['completion_date'])
df['duration_days'] = (df['completion_date'] - df['start_date']).dt.days
df = df[df['duration_days'] > 0] # drop any negative/invalid durations

In [23]:
print(df['duration_days'].describe())

count    247587.000000
mean        897.904943
std         886.683518
min           1.000000
25%         287.000000
50%         639.000000
75%        1222.000000
max       38562.000000
Name: duration_days, dtype: float64


In [24]:
df.nlargest(20, 'duration_days')[['nct_id', 'phase', 'start_date', 'completion_date', 'duration_days']]

,nct_id,phase,start_date,completion_date,duration_days
206007,NCT05343208,NA,1916-09-05,2022-04-04,38562
13457,NCT00257140,PHASE2/PHASE3,1931-06-30,1994-07-31,23042
139221,NCT03217539,NA,1977-07-07,2015-12-31,14056
67271,NCT01468883,PHASE3,1979-09-04,2016-11-17,13589
47657,NCT01014039,NA,1983-03-31,2017-01-31,12360
28539,NCT00591643,PHASE1,1977-07-31,2011-03-31,12296
137229,NCT03164291,PHASE3,1984-06-30,2017-05-18,12010
120166,NCT02719678,NA,1982-01-31,2014-09-30,11930
367,NCT00001197,PHASE2,1984-02-07,2015-05-18,11423
9146,NCT00165256,PHASE2,1995-05-15,2026-06-25,11364


Replace 'NA' with proper missing values:

In [25]:
print(df['phase'].value_counts())

phase
NA               126549
PHASE2            32303
PHASE1            29998
PHASE3            24710
PHASE4            20367
PHASE1/PHASE2      7233
PHASE2/PHASE3      3719
EARLY_PHASE1       2618
Name: count, dtype: int64


In [26]:
df['phase'] = df['phase'].replace('NA', np.nan)

Check missing values:

In [27]:
print(df.shape)
print(df.isna().sum())

(247587, 20)
nct_id                     0
phase                 126639
enrollment              1768
overall_status             0
start_date                 0
completion_date            0
sponsor_type               0
num_sites                  0
num_conditions             0
num_interventions          0
num_countries              0
allocation              2712
intervention_model      3111
masking                 2774
primary_purpose         4729
gender                   147
minimum_age            11107
maximum_age           104011
healthy_volunteers       596
duration_days              0
dtype: int64


Enrollment, allocation, intervention_model, masking, primary_purpose, gender, and healthy_volunteers have low missing values, so we can simply drop these rows:

In [30]:
df = df.dropna(subset=['enrollment', 'allocation', 'intervention_model', 'masking', 'primary_purpose', 'gender', 'healthy_volunteers'])
print(df.shape)
print(df.isna().sum())

(238041, 20)
nct_id                     0
phase                 123378
enrollment                 0
overall_status             0
start_date                 0
completion_date            0
sponsor_type               0
num_sites                  0
num_conditions             0
num_interventions          0
num_countries              0
allocation                 0
intervention_model         0
masking                    0
primary_purpose            0
gender                     0
minimum_age             9876
maximum_age            99733
healthy_volunteers         0
duration_days              0
dtype: int64


There is still the issue of the 120,000+ missing values in `phase`. These trials fall outside of the standard Phase 1-4 framework. We will drop them since `phase` will be a predictor for this analysis. This limitation should be acknowledged.

In [31]:
df = df.dropna(subset=['phase'])
print(df.shape)
print(df.isna().sum())

(114663, 20)
nct_id                    0
phase                     0
enrollment                0
overall_status            0
start_date                0
completion_date           0
sponsor_type              0
num_sites                 0
num_conditions            0
num_interventions         0
num_countries             0
allocation                0
intervention_model        0
masking                   0
primary_purpose           0
gender                    0
minimum_age            3274
maximum_age           49378
healthy_volunteers        0
duration_days             0
dtype: int64


We leave minimum_age and maximum_age alone and let LightGBM handle the nulls as the proportion of nulls is higher and there may be meaning to having null values. We are left with a cleaned dataset of 119,000+ completed interventional clinical trials.

Export the cleaned data:

In [32]:
df.to_csv('clinical_trials_cleaned.csv', index=False)